## Galaxy Cutout & Resize
Crop each galaxy to its main body then resize to 224×224.

In [ ]:
import os, glob
import numpy as np
import cv2
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from scipy.ndimage import median_filter, label as nd_label, binary_dilation
import matplotlib.pyplot as plt

input_folder = "test"
output_folder = "test_cutout"
TARGET = 224

os.makedirs(output_folder, exist_ok=True)


In [ ]:
def get_background(img):
    h, w = img.shape
    edge = min(40, h//8, w//8)
    corners = np.concatenate([
        img[:edge, :edge].flatten(),  img[:edge, -edge:].flatten(),
        img[-edge:, :edge].flatten(), img[-edge:, -edge:].flatten()
    ])
    _, med, std = sigma_clipped_stats(corners, sigma=2.5, maxiters=5)
    return med, std

def enhance_galaxy(img, bg, noise):
    cleaned = np.clip(img - bg, 0, None)
    s1 = median_filter(cleaned, size=3)
    s2 = median_filter(cleaned, size=5)
    return (s1 + s2) / 2

def find_radius(img, bg, noise):
    cy, cx = img.shape[0]//2, img.shape[1]//2
    enh = enhance_galaxy(img, bg, noise)
    thr = max(2.5*noise, np.percentile(enh, 75))
    mask = binary_dilation(enh > thr, iterations=2)
    labels, n = nd_label(mask)
    if n == 0:
        return 50
    best_lbl, best_score = 0, -1
    for i in range(1, n+1):
        region = labels == i
        if region.sum() < 50:
            continue
        ys, xs = np.where(region)
        dist = np.sqrt((xs.mean()-cx)**2 + (ys.mean()-cy)**2)
        score = enh[region].sum() / (1 + dist/10)
        if score > best_score:
            best_score, best_lbl = score, i
    if best_lbl == 0:
        return 50
    ys, xs = np.where(labels == best_lbl)
    dists = np.sqrt((xs-cx)**2 + (ys-cy)**2)
    return max(30, min(int(np.percentile(dists, 90)), min(img.shape)//2 - 10))

def smart_resize(img, size):
    interp = cv2.INTER_AREA if img.shape[0] > size else cv2.INTER_CUBIC
    return cv2.resize(img, (size, size), interpolation=interp)


In [ ]:
fits_files = glob.glob(os.path.join(input_folder, "*.fits"))
print(f"Found {len(fits_files)} files")

for i, fpath in enumerate(fits_files, 1):
    fname = os.path.basename(fpath)
    print(f"[{i}/{len(fits_files)}] {fname}")
    try:
        with fits.open(fpath) as hdul:
            img = hdul[0].data
            hdr = hdul[0].header

        bg, noise = get_background(img)
        radius    = find_radius(img, bg, noise)
        size      = min(int(radius * 2 * 1.4), min(img.shape))
        size     += size % 2

        pos = (img.shape[1]//2, img.shape[0]//2)
        cut = Cutout2D(img, pos, size, wcs=WCS(hdr))
        new_hdr = hdr.copy(); new_hdr.update(cut.wcs.to_header())

        fits.PrimaryHDU(cut.data, new_hdr).writeto(
            os.path.join(output_folder, f"cutout_{fname}"), overwrite=True)

        resized = smart_resize(cut.data.astype(np.float32), TARGET)
        fits.PrimaryHDU(resized, new_hdr).writeto(
            os.path.join(output_folder, f"resized_{fname}"), overwrite=True)

        print(f"  bg={bg:.4f}  noise={noise:.4f}  radius={radius}px  {size}x{size}->{TARGET}x{TARGET}")

        if i % 10 == 0:
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))
            for ax, d, title in zip(axes,
                [img, cut.data, resized],
                [f'{fname}\n{img.shape}', f'Cutout {size}x{size}', f'{TARGET}x{TARGET}']):
                v1, v2 = np.percentile(d, [1, 99])
                ax.imshow(d, cmap='gray', origin='lower', vmin=v1, vmax=v2)
                ax.set_title(title); ax.axis('off')
            plt.tight_layout()
            plt.savefig(os.path.join(output_folder,
                f"comp_{os.path.splitext(fname)[0]}.png"), dpi=150, bbox_inches='tight')
            plt.close()
            print(f"  → plot saved")
    except Exception as e:
        print(f"  Error: {e}")

print("\nDone!")
